In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("../data/raw/screentime_and_stress.csv")

In [3]:
# Define Input Features (X)
feature_cols = [
    'age',
    'occupation',
    'daily_screen_time_hours',
    'phone_usage_before_sleep_minutes',
    'sleep_duration_hours',
    'physical_activity_minutes'
]

# Define Target Variables (y)
target_cols = [
    'stress_level',
    'mental_fatigue_score'
]

X = df[feature_cols].copy()
y = df[target_cols].copy()


In [4]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42
)

### 🧮 Feature Construction

In [5]:
import sys
sys.path.append('..')  # lets the notebook import utils.py from project root

from utils import add_engineered_features

# Apply feature engineering 
X_train = add_engineered_features(X_train)
X_test = add_engineered_features(X_test)

print(X_train[['daily_screen_time_hours', 'phone_usage_before_sleep_minutes',
               'sleep_duration_hours', 'night_screen_ratio', 'rest_to_screen_ratio']].head())

       daily_screen_time_hours  phone_usage_before_sleep_minutes  \
9839                      5.20                                47   
9680                      3.90                                50   
7093                      1.06                                86   
11293                     7.33                               115   
820                       8.46                                94   

       sleep_duration_hours  night_screen_ratio  rest_to_screen_ratio  
9839                   6.33            0.150641              1.020968  
9680                   5.65            0.213675              1.153061  
7093                   8.87            1.352201              4.305825  
11293                  4.84            0.261482              0.581032  
820                    6.35            0.185185              0.671247  


### 🔀 Nominal Categorical Encoding


In [6]:
# Define categorical and numerical feature lists
categorical_cols = ['occupation']
numerical_cols = [
    'age',
    'daily_screen_time_hours',
    'phone_usage_before_sleep_minutes',
    'sleep_duration_hours',
    'physical_activity_minutes',
    'night_screen_ratio',
    'rest_to_screen_ratio'
]

In [7]:
# Encoding nominal category (occupation)
# Setup ColumnTransformer with OneHotEncoder
encoder = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

In [8]:
# Configure Scikit-Learn to output pandas DataFrames directly (preserves column names)
encoder.set_output(transform='pandas')

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

In [12]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(encoder, '../models/encoder.pkl')
print("Encoder saved as models/encoder.pkl")

Encoder saved as models/encoder.pkl


In [9]:
print(f"Original X_train shape: {X_train.shape}")
print(f"Encoded  X_train shape: {X_train_encoded.shape}")
print("\nEncoded Columns:")
print(X_train_encoded.columns.tolist())

Original X_train shape: (12000, 8)
Encoded  X_train shape: (12000, 14)

Encoded Columns:
['cat__occupation_Doctor', 'cat__occupation_Freelancer', 'cat__occupation_Manager', 'cat__occupation_Researcher', 'cat__occupation_Software Engineer', 'cat__occupation_Student', 'cat__occupation_Teacher', 'remainder__age', 'remainder__daily_screen_time_hours', 'remainder__phone_usage_before_sleep_minutes', 'remainder__sleep_duration_hours', 'remainder__physical_activity_minutes', 'remainder__night_screen_ratio', 'remainder__rest_to_screen_ratio']


### 📋 Demographics Benchmark & JSON Export

In [10]:
import importlib
import utils
importlib.reload(utils)
from utils import assign_age_group, compute_derived_productivity, to_percentage, \
                   STRESS_MIN, STRESS_MAX, PRODUCTIVITY_MIN, PRODUCTIVITY_MAX
import json


# Work on a copy of the full raw dataframe (not the train/test split)
bench_df = df.copy()

# Derive productivity from fatigue score
bench_df['derived_productivity'] = compute_derived_productivity(bench_df['mental_fatigue_score'])

# Convert both metrics to 0-100% scale
bench_df['stress_pct'] = bench_df['stress_level'].apply(
    lambda x: to_percentage(x, STRESS_MIN, STRESS_MAX)
)
bench_df['productivity_pct'] = bench_df['derived_productivity'].apply(
    lambda x: to_percentage(x, PRODUCTIVITY_MIN, PRODUCTIVITY_MAX)
)

# Bucket ages
bench_df['age_group'] = bench_df['age'].apply(assign_age_group)

# Aggregate per age group
benchmarks = bench_df.groupby('age_group').agg(
    avg_stress_pct=('stress_pct', 'mean'),
    avg_productivity_pct=('productivity_pct', 'mean'),
    avg_screen_time_hours=('daily_screen_time_hours', 'mean'),
    sample_size=('age', 'count')
).round(2)

print(benchmarks)

# Export as JSON, keyed by age_group
benchmark_dict = benchmarks.to_dict(orient='index')

with open('../data/processed/age_group_benchmarks.json', 'w') as f:
    json.dump(benchmark_dict, f, indent=4)

           avg_stress_pct  avg_productivity_pct  avg_screen_time_hours  \
age_group                                                                
18-32               66.31                 35.11                   5.47   
33-47               66.53                 34.60                   5.50   
48-59               66.51                 34.48                   5.54   

           sample_size  
age_group               
18-32             5293  
33-47             5517  
48-59             4190  


### Export processed data

In [11]:
import os

os.makedirs('../data/processed', exist_ok=True)

# Recombine encoded features with targets for saving
train_final = X_train_encoded.copy()
train_final['stress_level'] = y_train['stress_level'].values
train_final['mental_fatigue_score'] = y_train['mental_fatigue_score'].values

test_final = X_test_encoded.copy()
test_final['stress_level'] = y_test['stress_level'].values
test_final['mental_fatigue_score'] = y_test['mental_fatigue_score'].values

train_final.to_csv('../data/processed/train_cleaned_engineered_data.csv', index=False)
test_final.to_csv('../data/processed/test_cleaned_engineered_data.csv', index=False)

print(f"Train saved: {train_final.shape}")
print(f"Test saved: {test_final.shape}")

Train saved: (12000, 16)
Test saved: (3000, 16)
